In [2]:
import re
import time
from urllib.parse import urljoin, urlparse
import requests
from bs4 import BeautifulSoup
import pandas as pd

BASE = "https://lifehacker.ru/topics/technology/"
HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 "
        "(KHTML, like Gecko) Chrome/124.0 Safari/537.36"
    )
}
N_PAGES = 10
MAX_PER_PAGE = 30
REQUEST_TIMEOUT = 12
SLEEP = 0.5

session = requests.Session()
session.headers.update(HEADERS)

def get_soup(url: str) -> BeautifulSoup:
    r = session.get(url, timeout=REQUEST_TIMEOUT)
    r.raise_for_status()
    return BeautifulSoup(r.text, "html.parser")

def is_probably_article(href: str) -> bool:
    if not href:
        return False
    u = urlparse(href)
    if u.netloc and "lifehacker.ru" not in u.netloc:
        return False
    path = u.path or "/"
    bad_prefixes = (
        "/topics/", "/tag/", "/tags/", "/category/",
        "/author/", "/authors/", "/page/", "/pages/"
    )
    if path == "/" or any(path.startswith(bp) for bp in bad_prefixes):
        return False
    if re.search(r"\.(jpg|jpeg|png|gif|webp|svg|pdf)(\?|$)", path, re.I):
        return False
    segments = [s for s in path.split("/") if s]
    if len(segments) > 2:
        return False
    return True

def extract_list_links(soup: BeautifulSoup) -> list[str]:
    urls = []
    seen = set()

    for a in soup.select("main h2 a[href]"):
        href = a.get("href")
        if is_probably_article(href):
            full = urljoin(BASE, href)
            if full not in seen:
                seen.add(full)
                urls.append(full)

    if len(urls) < MAX_PER_PAGE:
        for a in soup.select("main a[href]"):
            href = a.get("href")
            if is_probably_article(href):
                full = urljoin(BASE, href)
                if full not in seen:
                    seen.add(full)
                    urls.append(full)
                    if len(urls) >= MAX_PER_PAGE:
                        break
    return urls[:MAX_PER_PAGE]

TITLE_SELECTORS = [
    "article h1", "header h1", "h1"
]
CONTENT_SELECTORS = [
    "article", ".entry-content", ".post__content",
    "div[itemprop='articleBody']", "main article", "main .content"
]

def extract_article(soup: BeautifulSoup) -> tuple[str, str]:
    # title
    title = ""
    for sel in TITLE_SELECTORS:
        node = soup.select_one(sel)
        if node:
            title = node.get_text(" ", strip=True)
            if title:
                break

    container = None
    for sel in CONTENT_SELECTORS:
        node = soup.select_one(sel)
        if node:
            container = node
            break
    if container is None:
        container = soup

    parts = []
    for p in container.select("p"):
        t = p.get_text(" ", strip=True)
        if t and len(t) > 3:
            parts.append(t)
    text = "\n\n".join(parts).strip()
    return title, text

records = []
seen_slugs = set()

for page in range(1, N_PAGES + 1):
    url = BASE if page == 1 else f"{BASE}?page={page}"
    print(f"[list] {url}")
    try:
        soup = get_soup(url)
    except Exception as e:
        print("  ! ошибка загрузки листинга:", e)
        continue

    links = extract_list_links(soup)
    print(f"  найдено ссылок: {len(links)}")
    time.sleep(SLEEP)

    for link in links:
        slug = urlparse(link).path.rstrip("/")
        if slug in seen_slugs:
            continue
        seen_slugs.add(slug)

        try:
            art = get_soup(link)
            title, text = extract_article(art)
            if title and text:
                records.append({"url": link, "title": title, "text": text})
                print(f"    [+] {title[:70]}…")
            else:
                print(f"    [-] пустой контент, пропуск: {link}")
        except Exception as e:
            print(f"    ! ошибка для {link}: {e}")
        time.sleep(SLEEP)

df = pd.DataFrame(records, columns=["url", "title", "text"])
print("\nСобрано материалов:", len(df))
df.head(10)

[list] https://lifehacker.ru/topics/technology/
  найдено ссылок: 30
    [+] Лучшее…
    [-] пустой контент, пропуск: https://lifehacker.ru/recipes/
    [+] лучшие проекты редакции…
    [-] пустой контент, пропуск: https://lifehacker.ru/health/
    [+] Реклама…
    [+] Для всех сезонов: Nike представила надувную куртку с регулируемой темп…
    [+] Суббренд Rivian представил велосипед-трансформер — он заряжается от кр…
    [+] Boox представила гибридный планшет и ридер-смартфон с цветными экранам…
    [+] 8 смартфонов с поддержкой AptX Lossless для тех, кто слышит разницу…
    [+] Обзор Honor X9d — неубиваемого смартфона с батарейкой 8 300 мА·ч…
    [+] Xiaomi показала Redmi TV X 2026 — 98-дюймовый телевизор с Mini-LED и ч…
    [+] Microsoft представила Mico — ИИ-версию легендарного «Скрепыша»…
    [+] В Microsoft Edge появился режим Copilot — теперь это ИИ-браузер…
    [+] Nike представила первую в мире «обувь с мотором» — как электровелосипе…
    [+] Samsung отложила запуск серии Gala

,url,title,text
0,https://lifehacker.ru/top/week/,Лучшее,Лучшее\n\nНовые комментарии\n\n0 / 0\n\n0 / 0\...
1,https://lifehacker.ru/special/services/,лучшие проекты редакции,Лайфхакер — это не только статьи. Ещё мы делае...
2,https://lifehacker.ru/reklama/,Реклама,Лайфхакер читает более 25 миллионов человек в ...
3,https://lifehacker.ru/predstavlena-naduvnaya-k...,Для всех сезонов: Nike представила надувную ку...,"Nike показала куртку Therma-FIT Air Milano, ко..."
4,https://lifehacker.ru/predstavlen-velosiped-also/,Суббренд Rivian представил велосипед-трансформ...,"Бренд Also, который раньше был частью Rivian, ..."
5,https://lifehacker.ru/anons-boox-note-air5-c-i...,Boox представила гибридный планшет и ридер-сма...,Компания Boox анонсировала два новых устройств...
6,https://lifehacker.ru/aptx-lossless-smartfony/,8 смартфонов с поддержкой AptX Lossless для те...,Беспроводной звук уже перестал быть синонимом ...
7,https://lifehacker.ru/obzor-honor-x9d/,Обзор Honor X9d — неубиваемого смартфона с бат...,Смартфоны в 2025 году стали обрастать ИИ-функц...
8,https://lifehacker.ru/anons-xiaomi-redmi-tv-x-...,Xiaomi показала Redmi TV X 2026 — 98-дюймовый ...,Xiaomi вместе со смартфонами Redmi K90 и часам...
9,https://lifehacker.ru/anons-mico/,Microsoft представила Mico — ИИ-версию легенда...,Microsoft официально представила Mico — анимир...
